In [1]:
from pathlib import Path

import pandas as pd
import torch


PROD = False

PATH_PLAYLIST = Path("../data/playlist")
PATH_LOOKUP = PATH_PLAYLIST / "track_lookup.parquet"
PATH_VOCAB = PATH_PLAYLIST / "training_vocab.parquet"
PATH_MODEL = Path("models") / "260301_vivid_dragon/model_t5M_ep7_v1d4984.pt"

# Loading model

In [2]:
device = torch.device('cpu')

ckpt     = torch.load(PATH_MODEL, map_location=device, weights_only=False)
hparams  = ckpt["hparams"]
vocab_ckpt = ckpt["vocab"]                                       # save before del

emb = ckpt["model_state_dict"]["embeddings_in.weight"].clone()   # (V, D), fp32
del ckpt                                                         # frees out-embedding + state dict
emb_norm = emb / emb.norm(dim=1, keepdim=True)

print(f"Embeddings: {emb.shape}  ({emb.element_size() * emb.numel() / 1e9:.2f} GB)")

Embeddings: torch.Size([5412049, 128])  (2.77 GB)


# Loading lookup

In [3]:
vocab = pd.DataFrame(vocab_ckpt)  # track_rowid, track_id — exactly what was used in training

lookup = pd.read_parquet(
    PATH_LOOKUP,
    columns=["track_rowid", "track_name", "artist_name", "track_popularity"],
)
lookup = lookup.merge(vocab, on="track_rowid", how="inner")
print(f"{len(lookup):,} tracks in lookup (should match vocab size {hparams['vocab_size']:,})")
lookup.head()

5,412,049 tracks in lookup (should match vocab size 5,412,049)


,track_rowid,track_name,artist_name,track_popularity,track_id
0,1,The Giver,Chappell Roan,89,0
1,2,Another Life,NOTD,43,1
2,3,Lover Online,NOTD,43,2
3,4,Crash,NOTD,56,3
4,5,I Just Missed A Call,NOTD,51,4


In [8]:
def find_neighbours(
    query: str,
    artist: str | None = None,
    k: int = 10,
    diverse: bool = True,
) -> pd.DataFrame:
    """Find top-k nearest neighbours by cosine similarity.

    `query` is matched case-insensitively against track_name.
    `artist` optionally narrows the match to a specific artist (case-insensitive contains).
    If multiple tracks still match, the most popular one is used.
    If `diverse=True`, at most one track per artist is returned in the results.
    """
    matches = lookup[lookup["track_name"].str.contains(query, case=False, na=False)]
    if artist is not None:
        matches = matches[matches["artist_name"].str.contains(artist, case=False, na=False)]
    if matches.empty:
        desc = f"'{query}'" + (f" by '{artist}'" if artist else "")
        print(f"No tracks matching {desc}")
        return pd.DataFrame()

    row = matches.sort_values("track_popularity", ascending=False).iloc[0]
    tid = int(row["track_id"])
    print(f"Query: {row['track_name']} — {row['artist_name']}  (pop {row['track_popularity']})")

    # topk avoids materialising the full vocab similarity array
    fetch = k * 50 + 1          # overfetch to survive diversity dedup; +1 for the query itself
    sims  = emb_norm[tid] @ emb_norm.T
    top_vals, top_idx = torch.topk(sims, min(fetch, len(sims)))

    sim_map    = dict(zip(top_idx.tolist(), top_vals.tolist()))
    candidates = lookup[lookup["track_id"].isin(sim_map)].copy()
    candidates["similarity"] = candidates["track_id"].map(sim_map)
    candidates = candidates[candidates["track_id"] != tid]
    candidates = candidates.sort_values("similarity", ascending=False)
    if diverse:
        candidates = candidates.drop_duplicates(subset="artist_name")
    return candidates.head(k)[["track_name", "artist_name", "track_popularity", "similarity"]]

In [44]:
queries = [
    ("Dunkelheit", "Burzum"),
]

for q in queries:
    display(find_neighbours(*q, k=10))

Query: Dunkelheit — Burzum  (pop 54)


,track_name,artist_name,track_popularity,similarity
3142088,Jesus' Tod,Burzum,48,0.967232
2751333,Transilvanian Hunger - Studio,Darkthrone,46,0.921476
2751179,Freezing Moon,Mayhem,55,0.875892
2607002,A Fine Day to Die,Bathory,48,0.858284
3961869,Radix Malorum,Gorgoroth,40,0.823646
1022756,I Troldskog Faren Vild,Ulver,40,0.822584
3678590,Withstand The Fall Of Time,Immortal,40,0.818786
3910033,The Blond Beast,Marduk,39,0.816876
2575248,Furrows of Gods - Борозни Богів,Drudkh,33,0.814720
4686592,Back to the Shadows,Minneriket,18,0.813043
